# Argus Vision — Train the Consensus Head

**Adversarial multi-agent visual debate for uncertainty-aware medical image classification.**

The consensus head is a small MLP that fuses both agents' probabilities, the spatial disagreement statistics, and the sentence embeddings of the two debate arguments into a single calibrated 8-class prediction.

It consumes the **788-dim** feature vector defined by the shared contract:

$$ z = [\,p_A(8)\;\|\;p_B(8)\;\|\;\text{spatial\_stats}(4)\;\|\;e_A(384)\;\|\;e_B(384)\,] $$

Pipeline:
1. Load both agents + the `all-MiniLM-L6-v2` argument encoder.
2. For every **training** image where the debate trigger fires: compute Grad-CAM++ (Agent A) and attention-rollout (Agent B) saliency, disagreement stats, then generate the round-1 and round-2 arguments via **Groq** — *cached to disk so repeated runs cost nothing*.
3. Build the 788-dim feature vector and the integer label.
4. Train the consensus MLP (`788->512->256->8`, BatchNorm + Dropout(0.3)) with AdamW, label smoothing 0.1, for 50 epochs.
5. Temperature-scale on the validation set (single scalar, LBFGS on NLL; ECE via `netcal`).
6. Save `consensus_best.pth` (state dict including the learned temperature) + the temperature value to `../backend/checkpoints`.

The MLP architecture below is the canonical definition shared with `backend/ml/consensus/classifier.py`.

## 1. Environment, constants and the shared contract

In [ ]:
import os
import subprocess
from pathlib import Path

# Install the pinned training stack plus scipy/groq used by the debate pipeline.
subprocess.run(
    ["pip", "install", "-q", "-r", "requirements_training.txt"],
    check=True,
)
subprocess.run(
    ["pip", "install", "-q", "scipy==1.13.0", "grad-cam==1.5.0",
     "opencv-python-headless==4.9.0.80", "groq==0.9.0", "netcal==1.3.5"],
    check=True,
)

import json  # noqa: E402
from typing import Any, Dict, List, Optional, Tuple  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
import torch.nn.functional as F  # noqa: E402

# --- Shared contract: ISIC-8 taxonomy (exact index 0..7 order) ----------
CLASS_NAMES: List[str] = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
FULL_NAMES: Dict[str, str] = {
    "MEL": "Melanoma",
    "NV": "Melanocytic Nevus",
    "BCC": "Basal Cell Carcinoma",
    "AK": "Actinic Keratosis",
    "BKL": "Benign Keratosis",
    "DF": "Dermatofibroma",
    "VASC": "Vascular Lesion",
    "SCC": "Squamous Cell Carcinoma",
}
NUM_CLASSES: int = 8
IMAGE_SIZE: int = 224
IMAGENET_MEAN: List[float] = [0.485, 0.456, 0.406]
IMAGENET_STD: List[float] = [0.229, 0.224, 0.225]
EMBEDDING_DIM: int = 384
FEATURE_DIM: int = 788  # 8 + 8 + 4 + 384 + 384.

# --- Debate-trigger thresholds (identical to the backend settings) ------
TAU_JS: float = 0.25
TAU_ENTROPY: float = 0.8

# --- Backbone identifiers (MUST match backend/ml/agents/*) --------------
AGENT_A_MODEL_NAME: str = "efficientnet_b4"
AGENT_B_MODEL_NAME: str = "vit_base_patch16_224.augreg_in21k_ft_in1k"

# --- Groq config (the LLM that realises the debate arguments) -----------
GROQ_MODEL: str = os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile")
GROQ_API_KEY: str = os.environ.get("GROQ_API_KEY", "")

# --- Training hyper-parameters ------------------------------------------
EPOCHS: int = 50
LR: float = 1e-3
WEIGHT_DECAY: float = 1e-4
BATCH_SIZE: int = 64
LABEL_SMOOTHING: float = 0.1
DROPOUT: float = 0.3

# --- Filesystem layout --------------------------------------------------
CHECKPOINT_DIR = Path("../backend/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
AGENT_A_CHECKPOINT = CHECKPOINT_DIR / "agent_a_best.pth"
AGENT_B_CHECKPOINT = CHECKPOINT_DIR / "agent_b_best.pth"
CONSENSUS_CHECKPOINT = CHECKPOINT_DIR / "consensus_best.pth"
TEMPERATURE_JSON = CHECKPOINT_DIR / "consensus_temperature.json"
ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE = ARTIFACT_DIR / "consensus_features.npz"
GROQ_CACHE_PATH = ARTIFACT_DIR / "groq_argument_cache.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")
print(f"Feature dimension: {FEATURE_DIM}")
print(f"Groq model: {GROQ_MODEL} (key set: {bool(GROQ_API_KEY)})")

## 2. The consensus MLP — canonical architecture

`ConsensusMLP` is the single source of truth for the consensus network. It is identical to `backend/ml/consensus/classifier.py`: three linear layers (`788->512->256->8`), with `BatchNorm1d` + `Dropout(0.3)` after each hidden layer, plus a learnable temperature scalar (`nn.Parameter`, init 1.0) that divides the logits before softmax for calibration.

In [ ]:
class ConsensusMLP(nn.Module):
    """Calibrated consensus classifier over the 788-dim debate feature vector.

    The network maps ``[pA(8), pB(8), spatial_stats(4), eA(384), eB(384)]`` to
    eight class logits via two hidden layers, each followed by ``BatchNorm1d``
    and ``Dropout``. A learnable scalar :attr:`temperature` divides the logits
    before softmax to calibrate the predictive distribution.

    Attributes:
        backbone: The ordered feature transform producing raw logits.
        temperature: A single learnable temperature scalar (init 1.0).
    """

    def __init__(
        self,
        input_dim: int = FEATURE_DIM,
        num_classes: int = NUM_CLASSES,
        dropout: float = DROPOUT,
    ) -> None:
        """Initialise the consensus MLP.

        Args:
            input_dim: Dimensionality of the input feature vector (788).
            num_classes: Number of output classes (8 for ISIC-8).
            dropout: Dropout probability applied after each hidden layer.
        """
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )
        self.temperature = nn.Parameter(torch.ones(1))

    def logits(self, features: torch.Tensor) -> torch.Tensor:
        """Return the raw (un-temperature-scaled) logits.

        Args:
            features: A ``(N, input_dim)`` feature batch.

        Returns:
            A ``(N, num_classes)`` tensor of raw logits.
        """
        return self.backbone(features)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """Return temperature-scaled logits ready for a softmax.

        Args:
            features: A ``(N, input_dim)`` feature batch.

        Returns:
            A ``(N, num_classes)`` tensor of logits divided by the temperature.
        """
        return self.logits(features) / self.temperature.clamp_min(1e-3)


print(ConsensusMLP())

## 3. Load the training/validation dataframes and the agents

We reproduce the same fixed-seed split as notebook 03 and load the two agents from `../backend/checkpoints`.

In [ ]:
import timm
from PIL import Image
from sklearn.model_selection import train_test_split
import torchvision.transforms as T

try:
    from config import AgentAConfig  # type: ignore
    from dataset import load_isic_dataframe  # type: ignore

    CFG = AgentAConfig()
    full_df = load_isic_dataframe(CFG)
    print(f"Loaded {len(full_df):,} labelled images via dataset.py.")
except Exception as exc:  # noqa: BLE001 - manual scan fallback.
    print(f"dataset.py loader unavailable ({exc}); scanning DATA_DIR.")
    DATA_DIR = Path(os.environ.get("ISIC_DATA_DIR", "./data"))
    gt_candidates = sorted(DATA_DIR.rglob("*GroundTruth*.csv"))
    if not gt_candidates:
        raise FileNotFoundError("No ISIC ground-truth CSV found.")
    raw = pd.read_csv(gt_candidates[0])
    image_col = raw.columns[0]
    onehot = raw[CLASS_NAMES].to_numpy()
    image_root = gt_candidates[0].parent
    records: List[Dict[str, object]] = []
    for i in range(len(raw)):
        name = str(raw.iloc[i][image_col])
        matches = list(image_root.rglob(f"{name}.jpg"))
        if matches:
            records.append(
                {"image_path": str(matches[0]), "label": int(onehot[i].argmax())}
            )
    full_df = pd.DataFrame.from_records(records)

full_df = full_df.reset_index(drop=True)
train_df, holdout_df = train_test_split(
    full_df, test_size=0.20, stratify=full_df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    holdout_df, test_size=0.50, stratify=holdout_df["label"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

eval_transform = T.Compose(
    [
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def load_tensor(image_path: str) -> torch.Tensor:
    """Load an image and return a preprocessed ``(1, 3, 224, 224)`` tensor.

    Args:
        image_path: Absolute path to a dermoscopic image on disk.

    Returns:
        A normalized float tensor with a leading batch dimension.
    """
    with Image.open(image_path) as handle:
        image = handle.convert("RGB")
    return eval_transform(image).unsqueeze(0)


def build_agent(model_name: str, checkpoint: Path) -> nn.Module:
    """Construct a timm backbone and load its fine-tuned checkpoint.

    Args:
        model_name: The timm model identifier for the backbone.
        checkpoint: Path to the agent's ``.pth`` state dict.

    Returns:
        The model in eval mode on ``DEVICE`` (ImageNet fallback if no ckpt).
    """
    has_ckpt = checkpoint.exists()
    model = timm.create_model(
        model_name, pretrained=not has_ckpt, num_classes=NUM_CLASSES
    )
    if has_ckpt:
        state = torch.load(checkpoint, map_location=DEVICE)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        model.load_state_dict(state, strict=False)
        print(f"Loaded checkpoint {checkpoint.name}.")
    else:
        print(f"WARNING: {checkpoint.name} missing; using ImageNet fallback.")
    return model.eval().to(DEVICE)


agent_a = build_agent(AGENT_A_MODEL_NAME, AGENT_A_CHECKPOINT)
agent_b = build_agent(AGENT_B_MODEL_NAME, AGENT_B_CHECKPOINT)
print("Agents ready.")

## 4. The argument encoder (`all-MiniLM-L6-v2`)

Each argument string is encoded into an L2-normalised 384-d vector — the same encoder and normalisation the backend `ArgumentEncoder` uses, so the cached embeddings are interchangeable with serving-time embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)


def encode_argument(text: str) -> np.ndarray:
    """Encode an argument string into a 384-d L2-normalised embedding.

    Args:
        text: The argument text to embed (empty text yields a zero-vector).

    Returns:
        A ``float32`` array of shape ``(EMBEDDING_DIM,)``.
    """
    if not text or not text.strip():
        return np.zeros(EMBEDDING_DIM, dtype=np.float32)
    vector = encoder.encode(text, normalize_embeddings=True)
    return np.asarray(vector, dtype=np.float32)


print("Encoder loaded; sample dim:", encode_argument("test argument").shape)

## 5. Attention, disagreement and trigger — mirrors of the backend modules

These helpers replicate `backend/ml/attention/{gradcam,rollout,disagreement}.py` and `backend/ml/debate/trigger.py`. The `spatial_stats` fed to the consensus head are `[mean_a, mean_b, std_a, std_b]` taken from the contested-region statistics (exactly as the contract specifies).

In [ ]:
import cv2
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from scipy.spatial.distance import jensenshannon

_EPS = 1e-12
GRID_SIZE = 14
NUM_PATCH_TOKENS = GRID_SIZE * GRID_SIZE


def shannon_entropy(probs: np.ndarray) -> float:
    """Compute the base-2 Shannon entropy (bits) of a probability vector."""
    p = np.clip(probs, 0.0, None)
    return -float(np.sum(p * np.log2(p + _EPS)))


def js_divergence(probs_a: np.ndarray, probs_b: np.ndarray) -> float:
    """Compute the squared Jensen-Shannon divergence between two vectors."""
    distance = jensenshannon(probs_a, probs_b, base=2)
    divergence = float(distance) ** 2
    return divergence if np.isfinite(divergence) else 0.0


def trigger_fires(div: float, ent_a: float, ent_b: float) -> bool:
    """Replicate the backend debate-trigger decision."""
    return (div > TAU_JS) or (max(ent_a, ent_b) > TAU_ENTROPY)


def gradcam_plusplus(model: nn.Module, tensor: torch.Tensor, target: int) -> np.ndarray:
    """Compute a 224x224 Grad-CAM++ saliency map for Agent A.

    Args:
        model: The EfficientNet-B4 backbone (Agent A).
        tensor: A ``(1, 3, 224, 224)`` input tensor on ``DEVICE``.
        target: The class index to explain.

    Returns:
        A ``float32`` ``(224, 224)`` map in ``[0, 1]``.
    """
    blocks = getattr(model, "blocks", None)
    if blocks is not None and len(blocks) > 0:
        last = blocks[-1]
        target_layer = getattr(last, "bn3", last)
    else:
        target_layer = getattr(model, "conv_head", None)
        if target_layer is None:
            for module in model.modules():
                if isinstance(module, (nn.BatchNorm2d, nn.Conv2d)):
                    target_layer = module
    model.zero_grad()
    cam = GradCAMPlusPlus(model=model, target_layers=[target_layer])
    grayscale = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(target)])
    model.zero_grad()
    heatmap = np.asarray(grayscale[0], dtype=np.float32)
    if heatmap.shape != (IMAGE_SIZE, IMAGE_SIZE):
        heatmap = cv2.resize(heatmap, (IMAGE_SIZE, IMAGE_SIZE),
                             interpolation=cv2.INTER_LINEAR)
    return np.clip(heatmap, 0.0, 1.0).astype(np.float32)


def attention_rollout(model: nn.Module, tensor: torch.Tensor) -> np.ndarray:
    """Compute a 224x224 attention-rollout saliency map for Agent B.

    Args:
        model: The ViT-B/16 backbone (Agent B).
        tensor: A ``(1, 3, 224, 224)`` input tensor on ``DEVICE``.

    Returns:
        A ``float32`` ``(224, 224)`` map in ``[0, 1]``.
    """
    handles: List[Any] = []
    captured: Dict[int, torch.Tensor] = {}
    original_fused: Dict[int, bool] = {}
    blocks = getattr(model, "blocks", None)

    def make_hook(layer_idx: int):
        def hook(module: nn.Module, inputs: tuple, output: torch.Tensor) -> None:
            x = inputs[0]
            batch, num_tokens, dim = x.shape
            heads = int(module.num_heads)
            head_dim = dim // heads
            scale = getattr(module, "scale", head_dim ** -0.5)
            qkv = module.qkv(x).reshape(batch, num_tokens, 3, heads, head_dim)
            qkv = qkv.permute(2, 0, 3, 1, 4)
            q, k = qkv[0], qkv[1]
            attn = (q @ k.transpose(-2, -1)) * float(scale)
            attn = attn.softmax(dim=-1)
            captured[layer_idx] = attn.mean(dim=1).detach().to(torch.float32).cpu()
        return hook

    try:
        for idx, block in enumerate(blocks):
            attn_module = getattr(block, "attn", None)
            if attn_module is None or not hasattr(attn_module, "qkv"):
                continue
            original_fused[idx] = bool(getattr(attn_module, "fused_attn", False))
            if hasattr(attn_module, "fused_attn"):
                attn_module.fused_attn = False
            handles.append(attn_module.register_forward_hook(make_hook(idx)))
        with torch.no_grad():
            model(tensor)
        layer_indices = sorted(captured.keys())
        num_tokens = captured[layer_indices[0]].shape[-1]
        identity = torch.eye(num_tokens, dtype=torch.float32)
        rollout = torch.eye(num_tokens, dtype=torch.float32)
        for idx in layer_indices:
            attn = captured[idx][0]
            aug = 0.5 * attn + 0.5 * identity
            aug = aug / aug.sum(dim=-1, keepdim=True).clamp_min(1e-12)
            rollout = aug @ rollout
        cls_attention = rollout[0, 1:1 + NUM_PATCH_TOKENS]
        grid = cls_attention.reshape(GRID_SIZE, GRID_SIZE).numpy().astype(np.float32)
        heatmap = cv2.resize(grid, (IMAGE_SIZE, IMAGE_SIZE),
                             interpolation=cv2.INTER_CUBIC).astype(np.float32)
        span = float(heatmap.max() - heatmap.min())
        if span < 1e-12:
            return np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
        return ((heatmap - heatmap.min()) / span).astype(np.float32)
    finally:
        for handle in handles:
            handle.remove()
        for idx, was_fused in original_fused.items():
            attn_module = getattr(blocks[idx], "attn", None)
            if attn_module is not None and hasattr(attn_module, "fused_attn"):
                attn_module.fused_attn = was_fused


def compute_disagreement(
    heatmap_a: np.ndarray, heatmap_b: np.ndarray
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """Compute contested-region statistics for both agents.

    Args:
        heatmap_a: Agent A's saliency map.
        heatmap_b: Agent B's saliency map.

    Returns:
        A ``(region_stats_a, region_stats_b)`` tuple, each a dict with
        ``mean``/``std``/``max`` floats inside the top-20%-mass contested mask.
    """
    def normalize(arr: np.ndarray) -> np.ndarray:
        arr = np.asarray(arr, dtype=np.float32)
        span = float(arr.max() - arr.min())
        if span < 1e-12:
            return np.zeros_like(arr, dtype=np.float32)
        return ((arr - arr.min()) / span).astype(np.float32)

    def region_stats(norm_map: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
        if not bool(mask.any()):
            return {"mean": 0.0, "std": 0.0, "max": 0.0}
        sel = norm_map[mask]
        return {"mean": float(sel.mean()), "std": float(sel.std()),
                "max": float(sel.max())}

    norm_a = normalize(heatmap_a)
    norm_b = normalize(heatmap_b)
    combined = norm_a + norm_b
    threshold = float(np.percentile(combined, 80.0))
    mask = combined >= threshold
    if not bool(mask.any()):
        mask = np.ones_like(combined, dtype=bool)
    return region_stats(norm_a, mask), region_stats(norm_b, mask)


print("Attention / disagreement / trigger helpers ready.")

## 6. Groq argument generation with on-disk caching

**Cost control is critical.** Every Groq response is cached to `artifacts/groq_argument_cache.json`, keyed by `image_id` (the image filename stem). If a key is present we **skip the API call entirely** and reuse the cached text. The cache is flushed to disk after every successful generation so an interrupted run never loses (or repeats) paid work.

When no `GROQ_API_KEY` is set, or any call fails, we degrade to a deterministic clinically grounded fallback so the feature build always completes — but those fallbacks are *not* cached, so a later run with a key will populate the cache properly.

In [ ]:
import re

# Two-sentence clinical descriptions (mirror of argument_gen.ISIC_CLASS_DESCRIPTIONS).
ISIC_CLASS_DESCRIPTIONS: Dict[str, str] = {
    "MEL": "Melanoma typically shows an atypical, broadened pigment network with "
           "irregular streaks, asymmetry of structure and colour, and frequent "
           "regression areas. A blue-white veil and chaotic vessels support malignancy.",
    "NV": "A melanocytic nevus is characterised by a symmetric, regularly spaced "
          "reticular or globular pattern with uniform colouration and a smooth "
          "transition to surrounding skin.",
    "BCC": "Basal cell carcinoma is defined by arborising (tree-like) vessels and "
           "blue-grey ovoid nests on a pigment-network-free background, with "
           "leaf-like areas and spoke-wheel structures.",
    "AK": "Actinic keratosis shows a 'strawberry' pattern: a red pseudo-network of "
          "dilated vessels around keratin-plugged follicular openings on a scaly, "
          "erythematous background.",
    "BKL": "Benign keratosis displays a cerebriform 'brain-like' surface with "
           "milia-like cysts and comedo-like openings, sharply demarcated borders "
           "and a stuck-on appearance.",
    "DF": "Dermatofibroma presents with a central white scar-like patch surrounded "
          "by a delicate peripheral pigment network and a homogeneous tan-brown ring.",
    "VASC": "Vascular lesions are recognised by sharply demarcated red, purple, or "
            "maroon lacunae separated by pale septa, with no melanocytic pigment network.",
    "SCC": "Squamous cell carcinoma shows central keratin masses, white circles "
           "around follicular openings, surface scale/ulceration, and looped or "
           "glomerular vessels at the periphery.",
}
_SYSTEM_PERSONA = (
    "You are a board-certified dermatology AI participating in a structured "
    "adversarial debate about a single dermoscopic skin-lesion image. Reason with "
    "the ABCDE rule and established dermoscopic criteria, citing specific visual "
    "evidence localised to the contested region. Argue concisely and never break "
    "character."
)
_DELTA_PATTERN = re.compile(r"CONFIDENCE_DELTA\s*:\s*([+-]?\d*\.?\d+)", re.IGNORECASE)

# Lazily construct a Groq client only when a key is present.
groq_client: Optional[Any] = None
if GROQ_API_KEY:
    try:
        from groq import Groq

        groq_client = Groq(api_key=GROQ_API_KEY)
        print("Groq client initialised.")
    except Exception as exc:  # noqa: BLE001
        print(f"Could not init Groq client ({exc}); using deterministic fallbacks.")
else:
    print("No GROQ_API_KEY set; using deterministic fallbacks (uncached).")

# Load (or initialise) the persistent Groq cache.
if GROQ_CACHE_PATH.exists():
    with open(GROQ_CACHE_PATH, "r", encoding="utf-8") as handle:
        GROQ_CACHE: Dict[str, Dict[str, Any]] = json.load(handle)
    print(f"Loaded {len(GROQ_CACHE)} cached debate transcripts.")
else:
    GROQ_CACHE = {}
    print("Starting with an empty Groq cache.")


def _flush_cache() -> None:
    """Persist the in-memory Groq cache to disk (called after every new entry)."""
    with open(GROQ_CACHE_PATH, "w", encoding="utf-8") as handle:
        json.dump(GROQ_CACHE, handle, indent=1)

In [ ]:
def _region_summary(stats: Dict[str, float]) -> str:
    """Render a one-line summary of a contested-region statistics dict."""
    return (
        f"In the contested region your attention map has mean "
        f"{stats.get('mean', 0.0):.3f}, std {stats.get('std', 0.0):.3f}, "
        f"peak {stats.get('max', 0.0):.3f}."
    )


def _fallback_argument(pred: str, conf: float, stats: Dict[str, float],
                       opponent: str) -> str:
    """Build a deterministic offline argument when the LLM is unavailable."""
    return (
        f"This lesion is most consistent with {FULL_NAMES[pred]} ({pred}) at "
        f"{conf * 100:.1f}% confidence. {ISIC_CLASS_DESCRIPTIONS[pred]} "
        f"{_region_summary(stats)} These features are inconsistent with "
        f"{FULL_NAMES.get(opponent, opponent)} ({opponent})."
    )


def _groq_complete(messages: List[Dict[str, str]]) -> Optional[str]:
    """Issue a single Groq chat-completion, returning text or None on failure."""
    if groq_client is None:
        return None
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL, messages=messages, temperature=0.4, max_tokens=300
        )
        content = response.choices[0].message.content
        return content.strip() if content and content.strip() else None
    except Exception as exc:  # noqa: BLE001
        print(f"Groq call failed ({exc}); using fallback.")
        return None


def run_debate(
    image_id: str,
    pred_a: str,
    conf_a: float,
    pred_b: str,
    conf_b: float,
    stats_a: Dict[str, float],
    stats_b: Dict[str, float],
) -> Dict[str, str]:
    """Produce (and cache) the two agents' round-1 and round-2 arguments.

    If ``image_id`` is already in the on-disk cache the cached transcript is
    returned untouched and **no** Groq request is made. Otherwise the four
    arguments (A/B x round 1/2) are generated, the cache is written to disk, and
    the transcript is returned. Fallback (offline) transcripts are not cached so
    a later keyed run can populate the cache properly.

    Args:
        image_id: Stable per-image cache key (the filename stem).
        pred_a: Agent A's predicted ISIC class code.
        conf_a: Agent A's confidence (0-1).
        pred_b: Agent B's predicted ISIC class code.
        conf_b: Agent B's confidence (0-1).
        stats_a: Agent A contested-region stats (mean/std/max).
        stats_b: Agent B contested-region stats (mean/std/max).

    Returns:
        A dict with keys ``arg_a_r1``, ``arg_b_r1``, ``arg_a_r2``, ``arg_b_r2``.
    """
    if image_id in GROQ_CACHE:
        return GROQ_CACHE[image_id]

    def opening(agent: str, pred: str, conf: float, stats: Dict[str, float],
                opp: str, opp_conf: float) -> str:
        messages = [
            {"role": "system", "content": _SYSTEM_PERSONA},
            {"role": "user", "content": (
                f"You are Agent {agent}, classifying this lesion as "
                f"{FULL_NAMES[pred]} ({pred}) at {conf * 100:.1f}% confidence. "
                f"Clinical profile: {ISIC_CLASS_DESCRIPTIONS[pred]} "
                f"{_region_summary(stats)} The opponent says {FULL_NAMES.get(opp, opp)} "
                f"({opp}) at {opp_conf * 100:.1f}%. In ONE paragraph argue why {pred} "
                f"is correct over {opp}, grounded in the contested-region evidence."
            )},
        ]
        text = _groq_complete(messages)
        return text if text is not None else _fallback_argument(pred, conf, stats, opp)

    def rebuttal(agent: str, pred: str, conf: float, stats: Dict[str, float],
                 own: str, opp_arg: str, opp: str, opp_conf: float) -> str:
        messages = [
            {"role": "system", "content": _SYSTEM_PERSONA},
            {"role": "user", "content": (
                f"You are Agent {agent}, defending {FULL_NAMES[pred]} ({pred}) at "
                f"{conf * 100:.1f}%. {_region_summary(stats)} Your opening argument: "
                f"\"{own}\". The opponent (predicting {FULL_NAMES.get(opp, opp)} / "
                f"{opp} at {opp_conf * 100:.1f}%) argued: \"{opp_arg}\". In ONE "
                f"paragraph rebut their strongest point, then on a final line output "
                f"'CONFIDENCE_DELTA: <number>' (a float in [-0.3, 0.3])."
            )},
        ]
        text = _groq_complete(messages)
        if text is None:
            text = _fallback_argument(pred, conf, stats, opp) + "\nCONFIDENCE_DELTA: 0.0"
        return text

    arg_a_r1 = opening("A", pred_a, conf_a, stats_a, pred_b, conf_b)
    arg_b_r1 = opening("B", pred_b, conf_b, stats_b, pred_a, conf_a)
    arg_a_r2 = rebuttal("A", pred_a, conf_a, stats_a, arg_a_r1, arg_b_r1, pred_b, conf_b)
    arg_b_r2 = rebuttal("B", pred_b, conf_b, stats_b, arg_b_r1, arg_a_r1, pred_a, conf_a)

    transcript = {
        "arg_a_r1": arg_a_r1,
        "arg_b_r1": arg_b_r1,
        "arg_a_r2": arg_a_r2,
        "arg_b_r2": arg_b_r2,
    }
    # Only cache genuine Groq output (skip when the client is unavailable so a
    # later keyed run can regenerate and populate the cache with real text).
    if groq_client is not None:
        GROQ_CACHE[image_id] = transcript
        _flush_cache()
    return transcript


def strip_delta(text: str) -> str:
    """Remove a trailing ``CONFIDENCE_DELTA`` line from an argument string."""
    return _DELTA_PATTERN.sub("", text).strip()


print("Debate-generation helpers ready.")

## 7. Build the 788-dim feature matrix over the triggering training images

For each **training** image we run both agents. Only images whose trigger fires get the full attention + debate treatment; the rest of the training set is omitted because the consensus head is invoked only on the debate path. The feature layout is exactly `[pA(8), pB(8), spatial_stats(4), eA(384), eB(384)]` with `spatial_stats = [mean_a, mean_b, std_a, std_b]`.

The full feature matrix is cached to `artifacts/consensus_features.npz` so a re-run skips both inference and Groq entirely.

In [ ]:
from tqdm.auto import tqdm


@torch.no_grad()
def predict_probs(model: nn.Module, tensor: torch.Tensor) -> np.ndarray:
    """Return the softmax probability vector for a single image tensor."""
    logits = model(tensor.to(DEVICE))
    return F.softmax(logits, dim=1)[0].detach().cpu().numpy().astype(np.float64)


def build_feature(
    pa: np.ndarray,
    pb: np.ndarray,
    stats_a: Dict[str, float],
    stats_b: Dict[str, float],
    emb_a: np.ndarray,
    emb_b: np.ndarray,
) -> np.ndarray:
    """Concatenate the 788-dim consensus feature vector.

    Args:
        pa: Agent A probabilities ``(8,)``.
        pb: Agent B probabilities ``(8,)``.
        stats_a: Agent A contested-region stats (mean/std/max).
        stats_b: Agent B contested-region stats (mean/std/max).
        emb_a: Agent A argument embedding ``(384,)``.
        emb_b: Agent B argument embedding ``(384,)``.

    Returns:
        A ``float32`` feature vector of shape ``(788,)``.
    """
    spatial_stats = np.array(
        [stats_a["mean"], stats_b["mean"], stats_a["std"], stats_b["std"]],
        dtype=np.float32,
    )
    feature = np.concatenate(
        [
            pa.astype(np.float32),
            pb.astype(np.float32),
            spatial_stats,
            emb_a.astype(np.float32),
            emb_b.astype(np.float32),
        ]
    )
    assert feature.shape == (FEATURE_DIM,), feature.shape
    return feature


def build_feature_matrix(frame: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """Build the consensus feature matrix over a frame's triggering images.

    Args:
        frame: A dataframe with ``image_path`` and integer ``label`` columns.

    Returns:
        A ``(features, labels)`` tuple of ``float32`` features ``(M, 788)`` and
        ``int64`` labels ``(M,)`` for the ``M`` images whose trigger fired.
    """
    features: List[np.ndarray] = []
    labels: List[int] = []
    for record in tqdm(frame.itertuples(index=False), total=len(frame)):
        image_path = str(record.image_path)
        image_id = Path(image_path).stem
        try:
            tensor = load_tensor(image_path)
        except Exception as exc:  # noqa: BLE001
            print(f"Skipping unreadable {image_path}: {exc}")
            continue
        pa = predict_probs(agent_a, tensor)
        pb = predict_probs(agent_b, tensor)
        if not trigger_fires(js_divergence(pa, pb), shannon_entropy(pa),
                              shannon_entropy(pb)):
            continue

        heatmap_a = gradcam_plusplus(agent_a, tensor, int(pa.argmax()))
        heatmap_b = attention_rollout(agent_b, tensor)
        stats_a, stats_b = compute_disagreement(heatmap_a, heatmap_b)

        transcript = run_debate(
            image_id, CLASS_NAMES[int(pa.argmax())], float(pa.max()),
            CLASS_NAMES[int(pb.argmax())], float(pb.max()), stats_a, stats_b,
        )
        emb_a = encode_argument(strip_delta(transcript["arg_a_r2"]))
        emb_b = encode_argument(strip_delta(transcript["arg_b_r2"]))

        features.append(build_feature(pa, pb, stats_a, stats_b, emb_a, emb_b))
        labels.append(int(record.label))
    return (
        np.asarray(features, dtype=np.float32),
        np.asarray(labels, dtype=np.int64),
    )


if FEATURE_CACHE.exists():
    cached = np.load(FEATURE_CACHE)
    X_train, y_train = cached["X_train"], cached["y_train"]
    X_val, y_val = cached["X_val"], cached["y_val"]
    print(f"Loaded cached features: train {X_train.shape}, val {X_val.shape}.")
else:
    print("Building TRAIN features...")
    X_train, y_train = build_feature_matrix(train_df)
    print("Building VAL features...")
    X_val, y_val = build_feature_matrix(val_df)
    np.savez_compressed(
        FEATURE_CACHE, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val
    )
    print(f"Saved features -> {FEATURE_CACHE.resolve()}")
print(f"Train features: {X_train.shape} | Val features: {X_val.shape}")

## 8. Train the consensus MLP (AdamW, 50 epochs, label smoothing 0.1)

We train on the contested training features with `CrossEntropyLoss(label_smoothing=0.1)` against the **raw** logits (the temperature is frozen at 1.0 during training and fitted separately afterwards). We track balanced accuracy on the held-out contested validation features and keep the best state dict.

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from torch.utils.data import DataLoader, TensorDataset

train_ds = TensorDataset(
    torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long()
)
val_ds = TensorDataset(
    torch.from_numpy(X_val).float(), torch.from_numpy(y_val).long()
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=len(train_ds) > BATCH_SIZE)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

consensus = ConsensusMLP().to(DEVICE)
# Freeze the temperature during MLP training; it is fitted post-hoc.
consensus.temperature.requires_grad = False
optimizer = torch.optim.AdamW(
    [p for p in consensus.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

history: Dict[str, List[float]] = {"loss": [], "val_bal_acc": []}
best_bal_acc = -1.0
best_state: Dict[str, torch.Tensor] = {}

for epoch in range(1, EPOCHS + 1):
    consensus.train()
    running = 0.0
    for features, targets in train_loader:
        features = features.to(DEVICE)
        targets = targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(consensus.logits(features), targets)
        loss.backward()
        optimizer.step()
        running += float(loss.item())
    scheduler.step()
    mean_loss = running / max(len(train_loader), 1)

    consensus.eval()
    preds: List[np.ndarray] = []
    truth: List[np.ndarray] = []
    with torch.no_grad():
        for features, targets in val_loader:
            logits = consensus.logits(features.to(DEVICE))
            preds.append(logits.argmax(dim=1).cpu().numpy())
            truth.append(targets.numpy())
    val_bal = float(balanced_accuracy_score(
        np.concatenate(truth), np.concatenate(preds)
    )) if preds else 0.0
    history["loss"].append(mean_loss)
    history["val_bal_acc"].append(val_bal)
    if val_bal >= best_bal_acc:
        best_bal_acc = val_bal
        best_state = {k: v.detach().cpu().clone()
                      for k, v in consensus.state_dict().items()}
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d} | loss {mean_loss:.4f} | val bal-acc {val_bal:.4f}")

print(f"Best validation balanced accuracy: {best_bal_acc:.4f}")
consensus.load_state_dict(best_state)

In [ ]:
import matplotlib.pyplot as plt

epochs_axis = np.arange(1, len(history["loss"]) + 1)
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
ax_loss.plot(epochs_axis, history["loss"], color="#3B7DD8")
ax_loss.set_title("Consensus training loss")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Cross-entropy (label smoothed)")
ax_acc.plot(epochs_axis, history["val_bal_acc"], color="#22C55E")
ax_acc.set_title("Validation balanced accuracy")
ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Balanced accuracy")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "consensus_training_curves.png", dpi=150)
plt.show()

## 9. Temperature-scaling calibration (LBFGS on NLL) + ECE

Holding the trained MLP fixed, we optimize a single temperature scalar $T$ to minimise the validation negative-log-likelihood of `softmax(logits / T)` using LBFGS. We then report the Expected Calibration Error before/after via `netcal`. The fitted $T$ is written back into `consensus.temperature` so the saved state dict already carries the calibration.

In [ ]:
from netcal.metrics import ECE

consensus.eval()
with torch.no_grad():
    val_logits = consensus.logits(torch.from_numpy(X_val).float().to(DEVICE))
val_targets = torch.from_numpy(y_val).long().to(DEVICE)

# A free temperature parameter optimised by LBFGS on the validation NLL.
temperature = nn.Parameter(torch.ones(1, device=DEVICE))
lbfgs = torch.optim.LBFGS([temperature], lr=0.01, max_iter=100)
nll = nn.CrossEntropyLoss()


def _closure() -> torch.Tensor:
    """LBFGS closure: NLL of the temperature-scaled validation logits."""
    lbfgs.zero_grad()
    loss = nll(val_logits / temperature.clamp_min(1e-3), val_targets)
    loss.backward()
    return loss


lbfgs.step(_closure)
fitted_temperature = float(temperature.detach().clamp_min(1e-3).item())
print(f"Fitted temperature T = {fitted_temperature:.4f}")

ece_metric = ECE(bins=15)
with torch.no_grad():
    probs_before = F.softmax(val_logits, dim=1).cpu().numpy()
    probs_after = F.softmax(
        val_logits / temperature.clamp_min(1e-3), dim=1
    ).cpu().numpy()
y_val_np = y_val.astype(np.int64)
ece_before = float(ece_metric.measure(probs_before, y_val_np))
ece_after = float(ece_metric.measure(probs_after, y_val_np))
print(f"ECE before calibration: {ece_before:.4f}")
print(f"ECE after  calibration: {ece_after:.4f}")

# Write the fitted temperature into the model parameter.
with torch.no_grad():
    consensus.temperature.copy_(temperature.detach().to(consensus.temperature.device))

## 10. Save `consensus_best.pth` + the temperature value

We persist the full state dict (which already contains `temperature` as a buffered `nn.Parameter`) to `../backend/checkpoints/consensus_best.pth`, plus a small JSON sidecar with the scalar temperature and the calibration metrics for easy inspection.

In [ ]:
torch.save(consensus.state_dict(), CONSENSUS_CHECKPOINT)
with open(TEMPERATURE_JSON, "w", encoding="utf-8") as handle:
    json.dump(
        {
            "temperature": fitted_temperature,
            "ece_before": ece_before,
            "ece_after": ece_after,
            "val_balanced_accuracy": best_bal_acc,
            "feature_dim": FEATURE_DIM,
            "train_examples": int(X_train.shape[0]),
            "val_examples": int(X_val.shape[0]),
        },
        handle,
        indent=1,
    )
print(f"Saved consensus state dict -> {CONSENSUS_CHECKPOINT.resolve()}")
print(f"Saved temperature sidecar -> {TEMPERATURE_JSON.resolve()}")

# Sanity check: reload into a fresh model and confirm the temperature survives.
reloaded = ConsensusMLP().to(DEVICE)
reloaded.load_state_dict(torch.load(CONSENSUS_CHECKPOINT, map_location=DEVICE))
print(f"Reloaded temperature: {float(reloaded.temperature.item()):.4f}")
assert abs(float(reloaded.temperature.item()) - fitted_temperature) < 1e-4
print("Consensus checkpoint verified.")

## 11. Summary

The calibrated consensus head is trained and saved. The Groq transcripts are cached in `artifacts/groq_argument_cache.json` (re-runs are free), and the 788-dim features are cached in `artifacts/consensus_features.npz`. Notebook **05** loads `consensus_best.pth` to evaluate the full Argus system against the agent and ensemble baselines.